<h1 style='text-align: center'><b><i>This notebook Collects and Cleans(preprocess) the data</b></h1>

<p style='text-align: center'><i>NOTE: Analysis will be done in different notebook</i></p>

**[Analysis notebook](Analysis.ipynb)**

#### **Major question**

> **Did economic ouput and income per person increase, and what can we reasonably infer about changes in people's material living standards?**

#### **Terms to get fimiliar with it first**

```text
1. GDP per capita vs GNI per capita
2. Inflation and purchasing power.
3. Population growth and per-capita measures
4. Mean, median, distribution and inequality 
```
Explaination of these terms we'll be added in `Terms_explainations.md` file

**[Terms_explainations.md](Terms_explainations.md)**



Instead of extracting every indicator upfront, we'll follow a **question-driven data analysis workflow**. 

- define the question,
- identify the required idicators,
- retrieve only the necessary data,
- analyze it, and
- interpret the results before moving on.

#### **Required modules**

In [29]:
import pandas as pd
import wbgapi as wb
import os

#### **Defining the data Collection Function**

It'll take a dictionary of **terms** and their **indicators**, time as a **range**, and economies ISO three-letter code e.g.(IND, CHN, USA etc) as its Parameters.

complete example:

```python 
datapoints: dict = {
    'GDP (current US$)' : 'NY.GDP.MKTP.CD',
    'GDP (constant 2015 US$)' : 'NY.GDP.MKTP.KD',
    ...
}

economy_code: str | list = 'IND' | ['IND', 'CHN', 'USA']

years: range = range(2004, 2024)
```

Now let's define the function...

In [3]:
def extractor(datapoints: dict, economy_code: str | list, years: range) -> pd.DataFrame:
    """Extract data from World Bank database, via (wbgapi) python package"""
    # Create a separate inidicators list.
    indicators_list: list = list(datapoints.values()) 

    # extracting engine
    df = wb.data.DataFrame(
        indicators_list,
        economy_code,
        time=years
    )

    return df

#### **Testing the `extractor` function**

In [15]:
datapoints: dict = {
    'GDP (current US$)' : 'NY.GDP.MKTP.CD',
    'GDP (constant 2015 US$)' : 'NY.GDP.MKTP.KD',
}

economy_code: str | list = 'IND' # for testing we'll just take India as economy

years: range = range(2004, 2025)

dataframe: pd.DataFrame = extractor(datapoints, economy_code, years)

dataframe

,YR2004,YR2005,YR2006,YR2007,YR2008,YR2009,YR2010,YR2011,YR2012,YR2013,...,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
series,,,,,,,,,,,,,,,,,,,,,
NY.GDP.MKTP.CD,7.091527e+11,8.203838e+11,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856722e+12,...,2.103588e+12,2.294797e+12,2.651474e+12,2.702930e+12,2.835606e+12,2.674852e+12,3.167271e+12,3.249938e+12,3.500906e+12,3.760813e+12
NY.GDP.MKTP.KD,1.013982e+12,1.094324e+12,1.182535e+12,1.273127e+12,1.312424e+12,1.415606e+12,1.535898e+12,1.616399e+12,1.704596e+12,1.813454e+12,...,2.103588e+12,2.277267e+12,2.432016e+12,2.588975e+12,2.689205e+12,2.533830e+12,2.779348e+12,2.990839e+12,3.206485e+12,3.434123e+12


The extractor function is working succefully

#### **Defining the data cleaning function**

take the **dataframe**, and the **datapoints dictionary** as its Parameters then clean and preprocess the dataframe.

_Note: This cleaning function is specifically designed for this particular project._


In [27]:
def dfcleaner(dataset: pd.DataFrame, datapoints: dict) -> pd.DataFrame:
    """Clean and preprocess the dataframe."""
    # Transposing the dataset
    dataset = dataset.T
    # Reseting it's index
    dataset = dataset.reset_index()
    # Renaming both axis labels to None
    dataset = dataset.rename_axis(None, axis=0).rename_axis(None, axis=1)

    # Now the most important step, assigning their columns name, for that
    # reverse the datapoints dictionary and assign it the rename function.
    reversed_datapoints: dict = {v:k for k, v in datapoints.items()}
    # Adding an extra index key and Year as it's value in reversed_datapoints.
    reversed_datapoints['index'] = 'Year'

    dataset = dataset.rename(columns=reversed_datapoints)

    # Final step cleaning the year column, we'll leave all of the other as it
    # is since this time we'll analyze through unrounded values.
    dataset['Year'] = dataset['Year'].str.replace('YR', '').astype(int)

    return dataset

#### **Testing the `dfcleaner` function**

In [28]:
cleaned_df: pd.DataFrame = dfcleaner(dataframe, datapoints)

cleaned_df.head()

,Year,GDP (current US$),GDP (constant 2015 US$)
0,2004,7.091527e+11,1.013982e+12
1,2005,8.203838e+11,1.094324e+12
2,2006,9.402599e+11,1.182535e+12
3,2007,1.216736e+12,1.273127e+12
4,2008,1.198895e+12,1.312424e+12


`dfcleaner` function is working fine.

#### **Let's create a main function**

takes the **datapoints dict**, **economy_code**, **time as range** and file paths from where the dataframe get's exported

In [51]:
def main_(datapoints:dict, economy_code: str | list, years:range, file_path: list):
    # Extract data with the help of extractor function
    dataframe: pd.DataFrame = extractor(datapoints,economy_code, years)
    # print(dataframe)
    # clean the data via df_cleaner function
    cleaned_df: pd.DataFrame = dfcleaner(dataframe, datapoints)
    # print(cleaned_df.head())

    try:
        cleaned_df.to_parquet(file_path[0])
        cleaned_df.to_csv(file_path[1])
        print('[SUCCESS] file has been saved to dedicated file_path')
    except Exception as e:
        print(f'[CRITICAL] exportation failed {e}')

#### **Let's test our final main function**

In [37]:
file_path: list = [
    'cleaned_datasets/Production/production_df.parquet',
    'cleaned_datasets/Stakeholders/stakeholders_df.csv'
]

datapoints: dict = {
    'GDP (current US$)' : 'NY.GDP.MKTP.CD',
    'GDP (constant 2015 US$)' : 'NY.GDP.MKTP.KD',
}

economy_code: str | list = 'IND' # for testing we'll just take India as economy

years: range = range(2004, 2025)

if __name__ == '__main__':
    main_(datapoints, economy_code, years, file_path)

[SUCCESS] file has been saved to dedicated file_path


well my data collection pipeline is **ready** to go.

### **Lets go for dedicated Data collection as per the required questions**

#### **Q1. How did GDP per capita change?**

1. **How much did GDP per capita inrease overall?**

2. **At what annual rate did GDP per capita grow?**

3. **Did GDP per capita grow consistently?**

4. **Was the increase real or mainly due to prices?**

#### **Selecting the required indicators**

| **Indicator** | **World Bank code** |
|---------------|---------------------|
| GDP per capita (current US$) | NY.GDP.PCAP.CD|
| GDP per capita (current LCU) | NY.GDP.PCAP.CN |
| GDP per capita (constant 2015 US$) | NY.GDP.PCAP.KD |
| GDP per capita (constant LCU) | NY.GDP.PCAP.KN |
| GDP per capita growth (annual %) | NY.GDP.PCAP.KD.ZG |


In [52]:
# Extracting the dataset.
file_path: list = [
    'cleaned_datasets/Production/df_ver01_production.parquet',
    'cleaned_datasets/Stakeholders/df_ver01_stakeholder.csv',
]

# Datapoints.
datapoints: dict = {
    'GDPPC(current US$)' : 'NY.GDP.PCAP.CD',
    'GDPPC(current LCU)' : 'NY.GDP.PCAP.CN',
    'GDPPC(constant 2015 US$)' : 'NY.GDP.PCAP.KD',
    'GDPPC(constant LCU)' : 'NY.GDP.PCAP.KN',
    'GDPPC_growth(annual %)' : 'NY.GDP.PCAP.KD.ZG'
}

economy_code: str = 'IND' # since we only need it for India

years: range = range(2004, 2025)

# Running the main function
if __name__ == '__main__':
    main_(datapoints, economy_code, years, file_path)

[SUCCESS] file has been saved to dedicated file_path


**Done collecting data for question 1**

Lets go for analysis face.

**[Analysis notebook](Analysis.ipynb)**

In [49]:
wb.series.info(q='GDP')

id,value
EG.GDP.PUSE.KO.PP,GDP per unit of energy use (PPP $ per kg of oil equivalent)
EG.GDP.PUSE.KO.PP.KD,GDP per unit of energy use (constant 2021 PPP $ per kg of oil equivalent)
EG.USE.COMM.GD.PP.KD,"Energy use (kg of oil equivalent) per $1,000 GDP (constant 2021 PPP)"
EN.GHG.CO2.RT.GDP.KD,Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)
EN.GHG.CO2.RT.GDP.PP.KD,Carbon intensity of GDP (kg CO2e per 2021 PPP $ of GDP)
NY.GDP.DEFL.KD.ZG,"Inflation, GDP deflator (annual %)"
NY.GDP.DEFL.KD.ZG.AD,"Inflation, GDP deflator: linked series (annual %)"
NY.GDP.DEFL.ZS,GDP deflator (base year varies by country)
NY.GDP.DEFL.ZS.AD,GDP deflator: linked series (base year varies by country)
NY.GDP.DISC.CN,Discrepancy in expenditure estimate of GDP (current LCU)
